# CaSR Altitude Correction File

Based on the CaSR temperature correction plan, the best way to correct the temperature is to use the lapse-rate of -6.5 K/km.

https://github.com/SalishSeaCast/analysis-junqi/blob/main/Analysis_Atmospheric_Forcing/Analysis_forcing_vs_observation/Temp_Correction_vs_Model.ipynb



## path and info

### Elevation data path and info

In [1]:
import xarray as xr

path_CaSR_GZ = (
    "/home/jqiu/analysis-junqi/"
    "Analysis_Atmospheric_Forcing/"
    "Analysis_results_comparison/"
    "00MMM2023/Data_CaSR_Altitude/"
    "CaSR_v3.2_P_GZ_SFC.nc"
)

ds_CaSR_GZ = xr.open_dataset(path_CaSR_GZ)

print(ds_CaSR_GZ)

<xarray.Dataset> Size: 119kB
Dimensions:             (rlon: 70, rlat: 105)
Coordinates:
    lon                 (rlat, rlon) float32 29kB ...
    lat                 (rlat, rlon) float32 29kB ...
  * rlon                (rlon) float64 560B -22.8 -22.71 ... -16.68 -16.59
  * rlat                (rlat) float64 840B -9.45 -9.36 -9.27 ... -0.18 -0.09
Data variables:
    rotated_pole        (rlon, rlat) int32 29kB ...
    CaSR_v3.2_P_GZ_SFC  (rlat, rlon) float32 29kB ...
Attributes:
    Conventions:  CF-1.6
    source:       https://hpfx.collab.science.gc.ca/~scar700/rcas-casr/
    institution:  Environment and Climate Change Canada / Environnement et Ch...
    title:        Canadian Surface Reanalysis (CaSR) / Réanalyse Canadienne d...
    contact:      rcas-casr@ec.gc.ca
    licence:      These data are produced and provided by the Environment and...


In [ ]:

var_name='CaSR_v3.2_P_GZ_SFC'
da=ds_CaSR_GZ[var_name]
# Variable


print("\n" + "=" * 80)
print(f"Variable: {var_name}")
print("=" * 80)

print(f"Dimensions : {da.dims}")
print(f"Shape      : {da.shape}")
print(f"Data type  : {da.dtype}")

for attr_name, attr_value in da.attrs.items():
    print(f"  {attr_name}: {attr_value}")




Variable: CaSR_v3.2_P_GZ_SFC
Dimensions : ('rlat', 'rlon')
Shape      : (105, 70)
Data type  : float32
  standard_name: Geopotential height
  long_name: Forecast: Geopotential height at surface
  units: dam
  grid_mapping: rotated_pole


### CaSR Forcing Field info

In [2]:
import xarray as xr

path_CaSR_example='/ocean/jqiu/forcings/CaSR/CaSR_y2023m03d01.nc'

ds_CaSR_example=xr.open_dataset(path_CaSR_example)

print(ds_CaSR_example)

<xarray.Dataset> Size: 6MB
Dimensions:       (time_counter: 24, y: 105, x: 70)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 192B 2023-03-01 ... 2023-03-0...
  * y             (y) int32 420B 0 1 2 3 4 5 6 7 ... 98 99 100 101 102 103 104
  * x             (x) int32 280B 0 1 2 3 4 5 6 7 8 ... 62 63 64 65 66 67 68 69
    nav_lat       (y, x) float32 29kB ...
    nav_lon       (y, x) float32 29kB ...
Data variables:
    solar         (time_counter, y, x) float32 706kB ...
    precip        (time_counter, y, x) float32 706kB ...
    therm_rad     (time_counter, y, x) float32 706kB ...
    tair          (time_counter, y, x) float32 706kB ...
    qair          (time_counter, y, x) float32 706kB ...
    u_wind        (time_counter, y, x) float32 706kB ...
    atmpres       (time_counter, y, x) float32 706kB ...
    v_wind        (time_counter, y, x) float32 706kB ...
    percentcloud  (time_counter, y, x) float32 706kB ...
Attributes:
    title:           Converted CaSR atmosphe

## Correction Construction

Altitude(dam, (rlon,rlat)) -> Altitude * 10 * (-6.5)/1000 -> Correction (y,x) (negative) 



In [6]:
path_output='/ocean/jqiu/forcings/CaSR_Temperature_Correction'

import xarray as xr

# Extract altitude (dam)
Altitude = ds_CaSR_GZ["CaSR_v3.2_P_GZ_SFC"]

# Calculate temperature correction (K)
Correction = Altitude * 10 * (-6.5 / 1000)

# Rename dimensions and variable
Correction = Correction.rename({
    "rlat": "y",
    "rlon": "x",
}).rename("Correction")

# Set attributes
# Units do not matter here, K or C both work.

Correction.attrs = {
    "long_name": "Temperature correction for CaSR based on surface altitude",
    "units": "K",
    "lapse_rate": "-6.5 K/km",
    "description": "Correction = altitude (dam) * 10 * (-6.5 / 1000)",
}



print(Correction)

ds_Correction = Correction.to_dataset()

ds_Correction.to_netcdf(path_output + "CaSR_Temperature_Correction.nc")

<xarray.DataArray 'Correction' (y: 105, x: 70)> Size: 29kB
array([[-0.92444414, -1.6162745 , -2.1906059 , ..., -6.4067802 ,
        -6.7017865 , -6.9496326 ],
       [-0.5610576 , -1.2162025 , -1.7764833 , ..., -6.2204437 ,
        -6.680688  , -6.6412916 ],
       [-0.24383074, -0.77449846, -1.3023556 , ..., -6.606778  ,
        -7.2000957 , -7.0258584 ],
       ...,
       [-0.        , -0.        , -0.        , ..., -5.022103  ,
        -5.621349  , -6.297927  ],
       [-0.        , -0.        , -0.        , ..., -5.794038  ,
        -6.270778  , -6.9438043 ],
       [-0.        , -0.        , -0.        , ..., -6.361929  ,
        -6.6344533 , -7.2494354 ]], shape=(105, 70), dtype=float32)
Coordinates:
    lon      (y, x) float32 29kB 235.9 236.0 236.1 236.2 ... 237.9 238.0 238.2
    lat      (y, x) float32 29kB 43.38 43.42 43.46 43.5 ... 54.42 54.46 54.5
  * x        (x) float64 560B -22.8 -22.71 -22.62 ... -16.77 -16.68 -16.59
  * y        (y) float64 840B -9.45 -9.36 -9.27 -9.1